In [1]:
import os
import sys
import numpy as np
import torch
import wandb
import argparse

# In notebooks, __file__ is not defined. Use os.getcwd() for the current directory.
module_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from hedging.envs import HedgeCallHeston
from experiments.utils.ppo_mlp_actor import create_ppo_mlp_actor
from experiments.utils.training_loop import action_training
from experiments.utils.testing import test_model
from experiments.utils.sim_config import (
    load_heston_data,
    EnvConfig,
    PPOConfig,
    TrainingConfig
)
import wandb
import torch
import warnings
import numpy as np

if not hasattr(torch.Tensor, "_orig_numpy"):
    torch.Tensor._orig_numpy = torch.Tensor.numpy

    def safe_numpy(self, *args, **kwargs):
        # transparently move tensor to CPU before numpy() if it's on GPU
        if self.is_cuda:
            warnings.warn("Calling .numpy() on CUDA tensor", stacklevel=2)
            return self.detach().cpu().numpy()
        return self._orig_numpy(*args, **kwargs)

    torch.Tensor.numpy = safe_numpy

from torchrl.envs import GymWrapper
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE


In [2]:
params, S0, K, v0 = load_heston_data("sp500")

In [11]:
def train_test_split(train_size: int, market: str):
    params, S0, K, v0 = load_heston_data(market=market)
    params_train = {}
    params_test = {}
    for key, value in params.items():
        params_train[key] = value[:train_size]
        params_test[key] = value[train_size:]
    
    S0_train = S0[:train_size]
    S0_test = S0[train_size:]

    K_train = K[:train_size]
    K_test = K[train_size:]

    v0_train = v0[:train_size]
    v0_test = v0[train_size:]

    train = (params_train, S0_train, K_train, v0_train)
    test = (params_test, S0_test, K_test, v0_test)

    return train, test

In [12]:
train, test = train_test_split(4, "sp500")

In [14]:
params, S0, K, v0 = train

In [15]:
S0

array([2257.253887, 2695.875134, 2510.564873, 3257.738735])

In [16]:
test

({'kappa': array([0.95516244]),
  'theta': array([0.08193806]),
  'rho': array([-0.95]),
  'sigma': array([0.50058473]),
  'lda': array([0.])},
 array([3699.63053]),
 array([[3680., 3690., 3700., 3710., 3720.]]),
 array([0.02713849]))